# Import Libraries
- yfinance as raw data
- torch as neural network & learning model
- pandas for dataframes 
- matplotlib for visualization
- sklearn for data preprocessing


In [ ]:
import yfinance as yf

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder

## Checking Cuda Viability

In [ ]:
# Printing if cuda is possible
print(torch.cuda.is_available())

# Printing the cuda device
print(torch.cuda.get_device_name(0))
print(torch.cuda.device_count())

# Setting cuda default device
torch.set_default_device('cuda')

# Checking if cuda is activated
print(torch.get_default_device())

# Importing Data from YFinance

In [ ]:
import yfinance as yf

ticker = "AAPL"

df = yf.download(ticker, period = "5y", interval = "1d")

In [ ]:
print(df.head())

# Reducing the dimensions
df.columns = df.columns.droplevel(1)

print(df.head())

In [ ]:
print(df.columns)

In [ ]:
# Defining transformation functions

# Simple Moving Average

def calculate_moving_average(dataframe, window):
    col_name = "".join(filter(str.isdigit, str(window)))

    dataframe[f"sma_{col_name}"] = (
        dataframe['Close']
        .rolling(window = window, min_periods = 1)
        .mean()
    )

    return dataframe; 

# RSI Calculations

def calculate_rsi(dataframe, window_size): 
    temp_df = pd.DataFrame({'Difference': dataframe['Close'].diff()})

    temp_df['Gain'] = temp_df['Difference'].clip(lower = 0)
    temp_df['Loss'] = temp_df['Difference'].clip(upper = 0).abs()

    temp_df['Avg_Gain'] = temp_df['Gain'].rolling(window = window_size, min_periods = 1).mean()
    temp_df['Avg_Loss'] = temp_df['Loss'].rolling(window = window_size, min_periods = 1).mean()

    rs = temp_df['Avg_Gain'] / temp_df['Avg_Loss']

    col_name = "".join(filter(str.isdigit, str(window_size)))

    dataframe[f'rsi_{col_name}'] = 100 - (100 / (1 + rs))

    return dataframe

def calculate_ewm_rsi(dataframe, window_size): 
    temp_df = pd.DataFrame({'Difference': dataframe['Close'].diff()})

    temp_df['Gain'] = temp_df['Difference'].clip(lower = 0)
    temp_df['Loss'] = temp_df['Difference'].clip(upper = 0).abs()

    temp_df['Avg_Gain'] = temp_df['Gain'].ewm(span = window_size, adjust = False).mean()
    
    temp_df['Avg_Loss'] = temp_df['Loss'].ewm(span = window_size, adjust = False).mean()

    rs = temp_df['Avg_Gain'] / temp_df['Avg_Loss']

    col_name = "".join(filter(str.isdigit, str(window_size)))

    dataframe[f'ewm_rsi_{col_name}'] = 100 - (100 / (1 + rs))

    return dataframe

# Volatility & Bollinger Bands
def calculate_volatility(dataframe, window_size): 
    col_name = "".join(filter(str.isdigit, str(window_size)))

    dataframe[f'volatility_{col_name}'] = dataframe['Close'].rolling(window = window_size, min_periods = 2).std()

    return dataframe

def calculate_bollinger_bands(dataframe, window_size, num_std): 
    roll_window = dataframe['Close'].rolling(window = window_size, min_periods = 2)

    mid_band = roll_window.mean()
    std_dev = roll_window.std()

    col_id = "".join(filter(str.isdigit, str(window_size)))

    dataframe[f'bollinger_band_mid_{col_id}'] = mid_band
    dataframe[f'bollinger_band_upper_{col_id}'] = mid_band + (num_std * std_dev)
    dataframe[f'bollinger_band_lower_{col_id}'] = mid_band - (num_std * std_dev)

    return dataframe

def transform(dataframe): 
    df = dataframe.sort_index().copy()
    
    df = calculate_moving_average(dataframe, 20)
    df = calculate_moving_average(dataframe, 50)
    df = calculate_moving_average(dataframe, 100)
    df = calculate_rsi(dataframe, 14)
    df = calculate_ewm_rsi(dataframe, 14)
    df = calculate_volatility(dataframe, 30)
    df = calculate_bollinger_bands(dataframe, 30, 2)

    return df

In [ ]:
prepared_df = transform(df)

In [ ]:
print(prepared_df.tail())

# Gold Data Preparation
### Main Adjustments: 
- Scaling features into pct_change values
- Clipping outliers after adjustments w/ np.percentile
- MinMaxScaler > StandardScaler
- Bollinger Band % instead of price based
- Encoding days of the week
- Dropping N/A & removing min_period values

### Reasoning: 
- Scaling features instead of using raw data allows the model to be trained based on scaling factors instead of hard values. These hard values make it hard to use MinMaxScaler & does not allow the LSTM model to learn from the data as it is ever increasing
- Outliers can force compress other vaules during scaling. 
- MinMaxScaler 

In [ ]:
def transform_gold(dataframe): 
    df = dataframe.copy()
    
    trimmed_df = df.drop(['High', 'Low', 'Open'], axis=1)
    
    # Assest that values match
    expected_columns = ['Close', 'Volume', 'sma_20', 'sma_50', 'sma_100',
                        'bollinger_band_upper_30', 'bollinger_band_lower_30',
                        'bollinger_band_mid_30', 'rsi_14', 'ewm_rsi_14', 'volatility_30']
    
    columns_to_drop = ['Close', 'Volume', 'sma_20', 'sma_50', 'sma_100', 'volatility_30', 
                        'day_of_week']
    
    is_match = set(trimmed_df.columns) == set(expected_columns)
    
    # Calculating target
    if not is_match:
        print("Columns do not match expected columns")
        return df
    
    trimmed_df['target'] = (trimmed_df['Close'] - trimmed_df['Close'].shift(1)) / trimmed_df['Close'].shift(1)
    
    target_lower = trimmed_df["target"].quantile(0.01)
    target_upper = trimmed_df["target"].quantile(0.99)
    
    trimmed_df["target"] = trimmed_df["target"].clip(lower=target_lower, upper=target_upper)
    
    # Calculating Price Relative MA's
    trimmed_df['close_to_sma_20'] = (trimmed_df['Close'] / trimmed_df['sma_20']) - 1
    trimmed_df['close_to_sma_50'] = (trimmed_df['Close'] / trimmed_df['sma_50']) - 1
    
    # Calculating Bollinger Bands
    trimmed_df["pc_b"] = (trimmed_df['Close'] - trimmed_df["bollinger_band_lower_30"]) / (trimmed_df['bollinger_band_upper_30'] - trimmed_df['bollinger_band_lower_30'])
    trimmed_df["pc_b"] = trimmed_df["pc_b"].clip(lower=0, upper=1)
    
    trimmed_df['bb_width'] = (trimmed_df['bollinger_band_upper_30'] - trimmed_df['bollinger_band_lower_30']) / trimmed_df['bollinger_band_mid_30']
    trimmed_df = trimmed_df.drop(columns=['bollinger_band_upper_30', 'bollinger_band_lower_30', 'bollinger_band_mid_30'])
    
    # Calculating RSI
    threshold = 0.85
    correlation = trimmed_df[['rsi_14', 'ewm_rsi_14']].corr()
    
    corr_value = correlation.iloc[0, 1]
    
    if abs(corr_value) > threshold: 
        columns_to_drop.append('ewm_rsi_14')
        print(f"RSI Correlation: {corr_value:.3f} - Dropping ewm_rsi_14 {abs(corr_value)} > threshold (0.85)")
        
    # Calculating Volatility
    pct_returns_series = trimmed_df['Close'].pct_change()
    trimmed_df['pct_volatility_30'] = pct_returns_series.rolling(window=30, min_periods=30).std()
    
    # Calculating Volume
    volume_pct_change = trimmed_df["Volume"].pct_change()
    trimmed_df['volume_pct_change'] = volume_pct_change
    
    vol_lower = trimmed_df['volume_pct_change'].quantile(0.01)
    vol_upper = trimmed_df['volume_pct_change'].quantile(0.99)
    
    trimmed_df['volume_pct_change'] = trimmed_df['volume_pct_change'].clip(lower=vol_lower, upper=vol_upper)
    
    # Calculating Cyclical Day of the Week
    trimmed_df['day_of_week'] = trimmed_df.index.dayofweek
    
    trimmed_df['day_sin'] = np.sin(2 * np.pi * trimmed_df['day_of_week'] / 5)
    trimmed_df['day_cos'] = np.cos(2 * np.pi * trimmed_df['day_of_week'] / 5)
    
    # Cleaning up 
    processed_df = trimmed_df.drop(columns=columns_to_drop)
    
    processed_df = processed_df.dropna()
    
    return processed_df

In [ ]:
gold_df = transform_gold(prepared_df)

print(gold_df)



# Splitting & Scaling The Data
### Changes From Norm: 
- Because stock data relies on inline data because of datetime indexing, splitting will be done without tts
- Splitting will be 70/15/15 for train/validate/test
- Will use MinMaxScaler 

In [ ]:
train_end = int(len(gold_df) * .70)
val_end = int(len(gold_df) * .85)

train_df = gold_df.iloc[:train_end]
validate_df = gold_df.iloc[train_end:val_end]
test_df = gold_df.iloc[val_end:]

X_train = train_df.drop(columns=['target'])
y_train = train_df['target'].values

X_validate = validate_df.drop(columns=['target'])
y_validate = validate_df['target'].values

X_test = test_df.drop(columns=['target'])
y_test = test_df['target'].values

print(len(train_df))
print(len(validate_df))
print(len(test_df))

In [ ]:
scaler = MinMaxScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_validate_scaled = scaler.transform(X_validate)
X_test_scaled = scaler.transform(X_test)

print(f"Train: {X_train_scaled.min()}")
print(f"Train: {X_train_scaled.max()}")

print(f"Validate: {X_validate_scaled.min()}")
print(f"Validate: {X_validate_scaled.max()}")

print(f"Test: {X_test_scaled.min()}")
print(f"Test: {X_test_scaled.max()}")

In [ ]:
print(f"X_train: {len(X_train_scaled)}")
print("------------")
print(f"y_train: {len(y_train)}")
print("------------")
print(f"X_validate: {len(X_validate_scaled)}")
print("------------")
print(f"y_validate: {len(y_validate)}")
print("------------")
print(f"X_test: {len(X_test_scaled)}")
print("------------")
print(f"y_test: {len(y_test)}")
print("------------")

In [ ]:
def create_sequence(features, target, window_size): 
    X = []
    y = []
    
    for i in range (len(features) - window_size): 
        window = features[i: i + window_size]
        label = target[i + window_size]
        
        X.append(window)
        y.append(label)
    
    return np.array(X), np.array(y)

In [ ]:
WINDOW_SIZE = 20

X_train, y_train = create_sequence(X_train_scaled, y_train, WINDOW_SIZE)
X_validate, y_validate = create_sequence(X_validate_scaled, y_validate, WINDOW_SIZE)
X_test, y_test = create_sequence(X_test_scaled, y_test, WINDOW_SIZE)

# Torchify It!

In [ ]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)

X_validate_tensor = torch.tensor(X_validate, dtype=torch.float32)
y_validate_tensor = torch.tensor(y_validate, dtype=torch.float32)

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
validate_dataset = TensorDataset(X_validate_tensor, y_validate_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=False)
validate_loader = DataLoader(validate_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
print(X_train_tensor.shape)

# Creating the Model

In [ ]:
INPUT_SIZE = 8
HIDDEN_SIZE = 64 # 4-8 times the size of your input size
NUM_LAYERS = 2 # Stacked LSTM layers
DROPOUT = 0.2

class StockLSTM(nn.Module): 
    def __init__(self, input_size, hidden_size, num_layers, dropout): 
        super(StockLSTM, self).__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.dropout = dropout
        
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, dropout=dropout, batch_first=True)
        self.regressor = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Dropout(dropout), 
            nn.Linear(32, 1)
        )
        
    def forward(self, x):
        lstm_out, (hidden, cell) = self.lstm(x)
        last_time_step = lstm_out[:, -1, :]
        
        prediction = self.regressor(last_time_step)
        
        return prediction

# Training Loop

In [ ]:
model = StockLSTM(INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS, DROPOUT)
criterion = nn.MSELoss()